In [1]:
# import os
# from google.colab import drive

# # 1. Mount to the STANDARD location (safest option)
# drive.mount('/content/drive')

# # 2. Define your paths clearly
# project_folder = "diffusers_csc2210"
# # This is where the repo sits (inside your Drive)
# working_dir = f"/content/drive/MyDrive/{project_folder}"
# parent_dir = "/content/drive/MyDrive/"

# # 3. Move to Google Drive Root first
# %cd $parent_dir

# # 4. Smart Clone/Update Logic
# if not os.path.exists(project_folder):
#     print(f"Cloning into {project_folder}...")
#     !git clone https://github.com/yyyyy7105/diffusers_csc2210.git
# else:
#     print(f"Repo exists. Updating...")
#     # FIX: You must enter the specific folder BEFORE pulling
#     %cd $project_folder
#     !git pull
#     # Go back to parent to keep the flow consistent for the next step
#     %cd $parent_dir

# # 5. Enter the repo for installation
# %cd $working_dir

# # 6. Install in editable mode
# print("Installing in editable mode...")
# !pip install -e .

# !pip install git+https://github.com/yyyyy7105/diffusers_csc2210
# !pip install bitsandbytes transformers pillow

In [2]:
# import os
# from google.colab import drive
# import shutil # Import shutil for rmtree

# # Check if /content/drive exists and is a directory
# if os.path.isdir('/content/drive'):
#     # If it's a directory but NOT a mount point, it means it's a local folder
#     # that might contain residual files. We need to remove it to allow a clean mount.
#     if not os.path.ismount('/content/drive'):
#         print("Removing non-mountpoint /content/drive directory to allow clean mount...")
#         shutil.rmtree('/content/drive')

# # Now attempt to mount. Using force_remount=True can help in cases where Drive was previously mounted
# # but became unresponsive, forcing a fresh mount.
# drive.mount('/content/drive', force_remount=True)

# cache_dir = "/content/drive/MyDrive/huggingface_cache"
# os.makedirs(cache_dir, exist_ok=True)
# os.environ['HF_HOME'] = cache_dir
# os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'  # Crucial for Google Drive
# cache_dir = None

In [3]:
import torch
from diffusers import Flux2KleinPipeline, PipelineQuantizationConfig
from transformers import BitsAndBytesConfig

device = "cuda"
print(f"{torch.cuda.is_available()=}")
dtype = torch.float16
enable_profiler = True
print(f"{enable_profiler=}")

# Configure 4-bit quantization using BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype
)

cache_dir = r"D:\Users\14623\Documents\Coding\ml\flux2\flux_cache\models--black-forest-labs--FLUX.2-klein-4B\snapshots\5e67da950fce4a097bc150c22958a05716994cea"

# Wrap BitsAndBytesConfig in PipelineQuantizationConfig explicitly
quantization_config = PipelineQuantizationConfig(quant_backend="bitsandbytes_4bit", quant_kwargs=bnb_config.to_dict())

pipe = Flux2KleinPipeline.from_pretrained(cache_dir, quantization_config=quantization_config, torch_dtype=dtype, enable_profiler=enable_profiler)
pipe.enable_model_cpu_offload()

torch.cuda.is_available()=True
enable_profiler=True


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
from PIL import Image
import torch


# Placeholder for your initial image. Replace with your actual image variable or path.
# For example: from PIL import Image; initial_image = Image.open("path/to/your/image.png")
# initial_image = Image.open("/content/Screenshot 2026-02-10 194248.png") # Please provide your initial image here. OPTIONAL

prompt = "better resolution, photo-realistic, a cat saying hello world"
image = pipe(
    prompt=prompt,
    # image=initial_image,  # OPTIONAL
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=4, # Since you are using 4 steps (likely Flux Schnell), keep this.
    generator=torch.Generator(device=device).manual_seed(0)
).images[0]
image.save("flux-klein.png")

I am running in: d:\Users\14623\Documents\gitRepo\diffusers_csc2210\2210
I am looking for logs at: d:\Users\14623\Documents\gitRepo\diffusers_csc2210\2210\log\flux2_profile


  0%|          | 0/4 [00:00<?, ?it/s]

-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
         1_check_inputs         0.06%     446.800us         0.06%     446.800us     446.800us             1  
        3_encode_prompt        31.57%     217.161ms       177.00%        1.218s        1.218s             1  
       4_process_images         0.00%      28.200us         0.00%      28.200us      28.200us             1  
      5_prepare_latents         0.06%     380.400us         0.44%       3.020ms       3.020ms             1  
    6_prepare_timesteps         0.05%     318.500us         0.09%     628.000us     628.000us             1  
       7_denoising_loop         0.15%       1.025ms       656.98%        4.520s        4.520s             1  
     trans

In [5]:
profile = pipe.get_profile()

---

# Phase 1: Attention Profiling & Visualization Tutorial

This section provides a **step-by-step walkthrough** to profile the attention layers of the Flux 2 Klein-4B transformer
and visualize the results. The goal is to identify exploitable structure (sparsity, locality, cross-modal patterns)
in the attention matrices across all 25 blocks and denoising timesteps.

**What you'll learn:**
1. How to enable the attention profiling mode
2. How to run inference while capturing attention metrics
3. How to visualize quadrant-level attention patterns (text↔text, text↔image, image↔image)
4. How to analyze entropy, sparsity, and head specialization
5. How to track attention pattern evolution across denoising timesteps

**Architecture reminder:**
- **5 double-stream blocks** (`Flux2TransformerBlock`) — joint attention between separate text and image streams
- **20 single-stream blocks** (`Flux2SingleTransformerBlock`) — self-attention on concatenated text+image tokens
- **24 attention heads**, 128 dimensions per head


## Step 1: Import Profiling Tools

Import the attention profiling data structures alongside the standard visualization libraries.
The key classes are:
- `AttentionProfileEntry` — stores metrics for one attention layer at one timestep
- `AttentionProfilingStore` — shared storage that computes and accumulates profiling data


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from diffusers.models.transformers.transformer_flux2 import (
    AttentionProfileEntry,
    AttentionProfilingStore,
)

print("Profiling tools imported successfully.")
print(f"Model has {len(pipe.transformer.transformer_blocks)} double-stream blocks")
print(f"Model has {len(pipe.transformer.single_transformer_blocks)} single-stream blocks")


## Step 2: Enable Attention Profiling

Call `set_profiling_mode()` on the transformer to swap all attention processors to **profiling variants**.
These processors compute `softmax(Q·Kᵀ/√d)` explicitly (detached, no grad) for metric recording,
while still using the efficient SDPA path for the actual computation — so **model output is unchanged**.

**Parameters:**
- `store_full_weights=False` — only stores summary statistics (entropy, sparsity, quadrant means). Set `True` to store full `[B, H, S, S]` weight matrices (⚠️ very memory-intensive).
- `sparsity_threshold=0.01` — fraction of weights below this value counts as "sparse"
- `topk=32` — number of top keys to measure concentration


In [ ]:
# Enable profiling — this swaps attention processors to profiling variants
pipe.transformer.set_profiling_mode(
    enabled=True,
    store_full_weights=False,  # Set True for full attention matrices (⚠️ uses lots of memory)
    sparsity_threshold=0.01,   # Threshold for sparsity measurement
    topk=32,                   # Number of top keys for concentration metric
)
print("✅ Attention profiling enabled on all 25 blocks.")


## Step 3: Run Inference with Profiling Enabled

Run the pipeline as normal. The profiling processors will capture attention metrics at every block
and every denoising timestep automatically.

**Tip:** Use diverse prompts to check if attention patterns are prompt-dependent.
Using `clear_attention_profile_data()` between prompts lets you tag entries per prompt.


In [ ]:
# Define prompts — use diverse types to check for prompt-dependent patterns
prompts = [
    "a photorealistic cat sitting on a wooden table",        # Simple object
    "a futuristic cityscape at sunset with flying cars",     # Complex scene
    "abstract swirling colors in a cosmic nebula",           # Abstract concept
]

all_entries = []  # Collect all profiling entries across prompts
generated_images = []  # Store generated images for display

for i, prompt in enumerate(prompts):
    # Clear previous profiling data before each prompt
    pipe.transformer.clear_attention_profile_data()
    print(f"\nPrompt {i+1}/{len(prompts)}: '{prompt}'")

    image = pipe(
        prompt=prompt,
        height=512,           # Use 512×512 for manageable profiling
        width=512,
        guidance_scale=1.0,
        num_inference_steps=4,
        generator=torch.Generator(device=device).manual_seed(42),
    ).images[0]

    # Retrieve captured profiling entries
    entries = pipe.transformer.get_attention_profile_data()
    print(f"  → Captured {len(entries)} profiling entries "
          f"({len(pipe.transformer.transformer_blocks)} double + "
          f"{len(pipe.transformer.single_transformer_blocks)} single blocks "
          f"× {4} timesteps)")

    # Tag entries with prompt index for later analysis
    for e in entries:
        e.prompt_index = i
    all_entries.extend(entries)
    generated_images.append(image)

# Display all generated images in a row
fig, axes = plt.subplots(1, len(generated_images), figsize=(5 * len(generated_images), 5))
if len(generated_images) == 1:
    axes = [axes]
for ax, img, p in zip(axes, generated_images, prompts):
    ax.imshow(img)
    ax.set_title(p[:40] + '...' if len(p) > 40 else p, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\n✅ Total profiling entries collected: {len(all_entries)}")


## Step 4: Organize Profiling Data

Group the collected entries by block and timestep for systematic analysis.
Each `AttentionProfileEntry` contains:
- `block_type` — "double" or "single"
- `block_index` — 0-4 for double, 0-19 for single
- `timestep` — the normalized denoising timestep
- `mean_weight_tt/ti/it/ii` — quadrant mean attention weights
- `per_head_entropy` — [H] tensor of entropy per head
- `per_head_sparsity` — [H] tensor of sparsity per head
- `per_head_topk_concentration` — [H] tensor of top-k concentration per head


In [ ]:
# Group entries by block and by timestep
entries_by_block = defaultdict(list)     # (block_type, block_index) -> [entries]
entries_by_timestep = defaultdict(list)   # rounded_timestep -> [entries]
unique_timesteps = sorted(set(e.timestep for e in all_entries))

for e in all_entries:
    entries_by_block[(e.block_type, e.block_index)].append(e)
    entries_by_timestep[round(e.timestep, 4)].append(e)

print(f"Unique timesteps: {len(unique_timesteps)} — {[f'{t:.4f}' for t in unique_timesteps]}")
print(f"Unique blocks: {len(entries_by_block)}")
print()
for key in sorted(entries_by_block.keys()):
    print(f"  {key[0]:>6} block {key[1]:>2}: {len(entries_by_block[key]):>3} entries")


## Step 5: Quadrant Analysis — Understanding Cross-Modal Attention

The attention matrix for a sequence `[text_tokens; image_tokens]` has **four quadrants**:

```
           text_K    img_K
text_Q  [ T→T       T→I   ]    ← text queries attending to text/image keys
img_Q   [ I→T       I→I   ]    ← image queries attending to text/image keys
```

**What to look for:**
- If **T→T is small** in later blocks → text self-attention may be redundant (maskable)
- If **I→I dominates** → image self-attention is the main computation (spatial locality masking candidate)
- If **T↔I is small** in some blocks → cross-modal attention may be skippable there


In [ ]:
def plot_quadrant_analysis(entries, title_suffix=""):
    """Plot mean attention weight in each quadrant across block depth."""
    double_entries = [e for e in entries if e.block_type == "double"]
    single_entries = [e for e in entries if e.block_type == "single"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for ax, block_entries, block_label in [
        (axes[0], double_entries, "Double-Stream (Joint Attn)"),
        (axes[1], single_entries, "Single-Stream (Self-Attn)"),
    ]:
        if not block_entries:
            ax.set_title(f"{block_label} — No data")
            continue

        block_indices = sorted(set(e.block_index for e in block_entries))
        quadrants = {"T→T": [], "T→I": [], "I→T": [], "I→I": []}

        for idx in block_indices:
            block_e = [e for e in block_entries if e.block_index == idx]
            quadrants["T→T"].append(np.mean([e.mean_weight_tt for e in block_e]))
            quadrants["T→I"].append(np.mean([e.mean_weight_ti for e in block_e]))
            quadrants["I→T"].append(np.mean([e.mean_weight_it for e in block_e]))
            quadrants["I→I"].append(np.mean([e.mean_weight_ii for e in block_e]))

        x = np.arange(len(block_indices))
        width = 0.2
        colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

        for i, (label, values) in enumerate(quadrants.items()):
            ax.bar(x + i * width, values, width, label=label, color=colors[i])

        ax.set_xlabel("Block Index")
        ax.set_ylabel("Mean Attention Weight")
        ax.set_title(f"{block_label}{title_suffix}")
        ax.set_xticks(x + 1.5 * width)
        ax.set_xticklabels(block_indices)
        ax.legend()

    plt.tight_layout()
    plt.show()

# Plot overall quadrant analysis (averaged across all prompts and timesteps)
print("=== Overall Quadrant Analysis ===")
plot_quadrant_analysis(all_entries, " — Averaged Across All Prompts & Timesteps")


In [ ]:
# Plot quadrant analysis at individual timesteps to see temporal variation
print("=== Quadrant Analysis Per Timestep ===")
# Sample evenly across timesteps (show ~4 timesteps)
sample_indices = list(range(0, len(unique_timesteps), max(1, len(unique_timesteps) // 4)))
for step_idx in sample_indices:
    t = unique_timesteps[step_idx]
    t_round = round(t, 4)
    t_entries = entries_by_timestep.get(t_round, [])
    if t_entries:
        plot_quadrant_analysis(t_entries, f" — Timestep t={t_round:.4f}")


## Step 6: Entropy & Sparsity Profiles

These metrics reveal which blocks have the most structured (exploitable) attention patterns:

- **Entropy** — lower entropy = more concentrated attention = more maskable
- **Sparsity** — higher sparsity = more near-zero weights = more prunable
- **Top-k concentration** — higher = attention is focused on fewer keys

Blocks with **high sparsity** and **low entropy** are the best candidates for attention masking.


In [ ]:
def plot_entropy_sparsity(entries, title_suffix=""):
    """Plot per-block entropy, sparsity, and top-k concentration."""
    double_entries = [e for e in entries if e.block_type == "double"]
    single_entries = [e for e in entries if e.block_type == "single"]

    all_blocks = []
    all_labels = []

    for e_list, prefix in [(double_entries, "D"), (single_entries, "S")]:
        block_indices = sorted(set(e.block_index for e in e_list))
        for idx in block_indices:
            block_e = [e for e in e_list if e.block_index == idx]
            mean_entropy = torch.stack([e.per_head_entropy for e in block_e]).mean().item()
            mean_sparsity = torch.stack([e.per_head_sparsity for e in block_e]).mean().item()
            mean_topk = torch.stack([e.per_head_topk_concentration for e in block_e]).mean().item()
            all_blocks.append((mean_entropy, mean_sparsity, mean_topk))
            all_labels.append(f"{prefix}{idx}")

    if not all_blocks:
        print("No data to plot.")
        return

    entropies, sparsities, topks = zip(*all_blocks)
    x = np.arange(len(all_labels))

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    ax1.bar(x, entropies, color="#2196F3", alpha=0.8)
    ax1.set_ylabel("Mean Entropy")
    ax1.set_title(f"Per-Block Attention Entropy{title_suffix}")
    # Mark boundary between double and single blocks
    n_double = len(set(e.block_index for e in double_entries))
    if n_double > 0:
        ax1.axvline(x=n_double - 0.5, color="gray", linestyle="--", alpha=0.5,
                   label="Double→Single boundary")
        ax1.legend()

    ax2.bar(x, sparsities, color="#F44336", alpha=0.8)
    ax2.set_ylabel("Sparsity (frac < threshold)")
    ax2.set_title(f"Per-Block Attention Sparsity{title_suffix}")

    ax3.bar(x, topks, color="#4CAF50", alpha=0.8)
    ax3.set_ylabel("Top-k Concentration")
    ax3.set_title(f"Per-Block Top-k Attention Concentration{title_suffix}")
    ax3.set_xticks(x)
    ax3.set_xticklabels(all_labels, rotation=45, ha="right")
    ax3.set_xlabel("Block (D=Double, S=Single)")

    plt.tight_layout()
    plt.show()

plot_entropy_sparsity(all_entries, " — Averaged Across All Prompts & Timesteps")


## Step 7: Per-Head Analysis — Do Individual Heads Specialize?

Check if different attention heads within a block specialize in different patterns.
If some heads are consistently high-entropy while others are low-entropy,
we may be able to selectively mask only specific heads.


In [ ]:
def plot_per_head_analysis(entries, block_type="double", block_index=0):
    """Plot per-head entropy, sparsity, and top-k for a specific block."""
    block_entries = [e for e in entries
                     if e.block_type == block_type and e.block_index == block_index]
    if not block_entries:
        print(f"No data for {block_type} block {block_index}")
        return

    # Average across timesteps and prompts
    entropies = torch.stack([e.per_head_entropy for e in block_entries]).mean(dim=0)
    sparsities = torch.stack([e.per_head_sparsity for e in block_entries]).mean(dim=0)
    topks = torch.stack([e.per_head_topk_concentration for e in block_entries]).mean(dim=0)

    num_heads = entropies.shape[0]
    x = np.arange(num_heads)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

    ax1.bar(x, entropies.numpy(), color="#2196F3")
    ax1.set_xlabel("Head Index")
    ax1.set_ylabel("Entropy")
    ax1.set_title(f"Per-Head Entropy — {block_type} block {block_index}")

    ax2.bar(x, sparsities.numpy(), color="#F44336")
    ax2.set_xlabel("Head Index")
    ax2.set_ylabel("Sparsity")
    ax2.set_title(f"Per-Head Sparsity — {block_type} block {block_index}")

    ax3.bar(x, topks.numpy(), color="#4CAF50")
    ax3.set_xlabel("Head Index")
    ax3.set_ylabel("Top-k Concentration")
    ax3.set_title(f"Per-Head Top-k — {block_type} block {block_index}")

    plt.tight_layout()
    plt.show()

# Analyze first and last double-stream blocks
num_double = len(pipe.transformer.transformer_blocks)
num_single = len(pipe.transformer.single_transformer_blocks)

print("=== Double-Stream Blocks (First & Last) ===")
for idx in [0, num_double - 1]:
    plot_per_head_analysis(all_entries, "double", idx)

print("\n=== Single-Stream Blocks (First, Middle, Last) ===")
for idx in [0, num_single // 2, num_single - 1]:
    plot_per_head_analysis(all_entries, "single", idx)


## Step 8: Temporal Evolution — How Attention Changes During Denoising

This shows how attention patterns change across the denoising process:
- **Early timesteps** (high noise): Attention should be more diffuse (higher entropy)
- **Late timesteps** (low noise): Attention should be more focused (lower entropy, higher sparsity)

If patterns change significantly across timesteps, we may need **timestep-adaptive masking**.


In [ ]:
def plot_temporal_evolution(entries):
    """Plot entropy and I→I weight evolution across denoising timesteps."""
    timesteps = sorted(set(e.timestep for e in entries))
    if len(timesteps) < 2:
        print("Need at least 2 timesteps for temporal evolution plot.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for ax_row, block_type, label in [
        (axes[0], "double", "Double-Stream"),
        (axes[1], "single", "Single-Stream"),
    ]:
        block_entries = [e for e in entries if e.block_type == block_type]
        if not block_entries:
            continue

        block_indices = sorted(set(e.block_index for e in block_entries))

        # Left: Entropy vs. Timestep
        ax = ax_row[0]
        for idx in block_indices:
            t_vals, e_vals = [], []
            for t in timesteps:
                te = [e for e in block_entries
                      if e.block_index == idx and abs(e.timestep - t) < 1e-6]
                if te:
                    t_vals.append(t)
                    e_vals.append(np.mean([e.per_head_entropy.mean().item() for e in te]))
            if t_vals:
                ax.plot(t_vals, e_vals, marker='o', label=f'Block {idx}', markersize=3)
        ax.set_xlabel('Timestep')
        ax.set_ylabel('Mean Entropy')
        ax.set_title(f'{label} — Entropy vs. Timestep')
        ax.legend(fontsize=7)

        # Right: I→I Weight vs. Timestep
        ax = ax_row[1]
        for idx in block_indices:
            t_vals, ii_vals = [], []
            for t in timesteps:
                te = [e for e in block_entries
                      if e.block_index == idx and abs(e.timestep - t) < 1e-6]
                if te:
                    t_vals.append(t)
                    ii_vals.append(np.mean([e.mean_weight_ii for e in te]))
            if t_vals:
                ax.plot(t_vals, ii_vals, marker='o', label=f'Block {idx}', markersize=3)
        ax.set_xlabel('Timestep')
        ax.set_ylabel('Mean I→I Weight')
        ax.set_title(f'{label} — I→I Weight vs. Timestep')
        ax.legend(fontsize=7)

    plt.tight_layout()
    plt.show()

plot_temporal_evolution(all_entries)


## Step 9: Summary Statistics Table

A comprehensive numerical summary of all profiling metrics, useful for deciding
which blocks and strategies to prioritize for Phase 2 masking experiments.


In [ ]:
def print_summary_table(entries):
    """Print a formatted summary table of profiling metrics."""
    header = (f"{'Type':<8} {'Block':<6} {'#Entries':<9} "
              f"{'T→T':>8} {'T→I':>8} {'I→T':>8} {'I→I':>8} "
              f"{'Entropy':>8} {'Sparsity':>9} {'Top-k':>8}")
    print(header)
    print('─' * len(header))

    for block_type in ['double', 'single']:
        block_entries = [e for e in entries if e.block_type == block_type]
        block_indices = sorted(set(e.block_index for e in block_entries))

        for idx in block_indices:
            be = [e for e in block_entries if e.block_index == idx]
            n = len(be)
            tt = np.mean([e.mean_weight_tt for e in be])
            ti = np.mean([e.mean_weight_ti for e in be])
            it = np.mean([e.mean_weight_it for e in be])
            ii = np.mean([e.mean_weight_ii for e in be])
            ent = np.mean([e.per_head_entropy.mean().item() for e in be])
            sp = np.mean([e.per_head_sparsity.mean().item() for e in be])
            tk = np.mean([e.per_head_topk_concentration.mean().item() for e in be])
            print(f"{block_type:<8} {idx:<6} {n:<9} "
                  f"{tt:>8.5f} {ti:>8.5f} {it:>8.5f} {ii:>8.5f} "
                  f"{ent:>8.3f} {sp:>9.4f} {tk:>8.4f}")

        if block_type == 'double' and block_entries:
            print('─' * len(header))

print_summary_table(all_entries)


## Step 10: Save Profiling Results

Save the profiling data for offline analysis and for use in Phase 2 masking experiments.
Data is saved in two formats:
- **Pickle** — full fidelity, includes tensor data
- **CSV** — quick inspection in spreadsheets


In [ ]:
import pickle, csv, os

output_dir = "2210/profiling_results"
os.makedirs(output_dir, exist_ok=True)

# Save as pickle (full fidelity)
pkl_path = os.path.join(output_dir, "attention_profile_entries.pkl")
with open(pkl_path, 'wb') as f:
    pickle.dump(all_entries, f)
print(f"Saved {len(all_entries)} entries to {pkl_path}")

# Save CSV summary
csv_path = os.path.join(output_dir, "attention_profile_summary.csv")
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'block_type', 'block_index', 'timestep',
        'num_text_tokens', 'num_image_tokens',
        'mean_weight_tt', 'mean_weight_ti', 'mean_weight_it', 'mean_weight_ii',
        'mean_entropy', 'mean_sparsity', 'mean_topk_concentration',
    ])
    for e in all_entries:
        writer.writerow([
            e.block_type, e.block_index, e.timestep,
            e.num_text_tokens, e.num_image_tokens,
            e.mean_weight_tt, e.mean_weight_ti, e.mean_weight_it, e.mean_weight_ii,
            e.per_head_entropy.mean().item(),
            e.per_head_sparsity.mean().item(),
            e.per_head_topk_concentration.mean().item(),
        ])
print(f"Saved CSV summary to {csv_path}")


## Step 11: Disable Profiling & Restore Normal Mode

After profiling is complete, disable it to restore the standard (efficient) attention
processors. This is important for normal inference performance.


In [ ]:
pipe.transformer.set_profiling_mode(enabled=False)
print("✅ Attention profiling disabled. Standard attention processors restored.")
print("   You can now use the pipeline for normal inference at full speed.")


## Interpreting Results & Next Steps

### What to look for in the profiling results:

| Pattern Observed | Implication | Masking Strategy |
|---|---|---|
| T→T small in later double blocks | Text self-attention is redundant | Strategy A1: Block T→T |
| I→I dominates in single blocks | Image self-attention is the bottleneck | Strategy B: Spatial locality masking |
| High sparsity in last N single blocks | Later blocks have prunable attention | Strategy D1: Mask last N blocks |
| Entropy varies across timesteps | Patterns are timestep-dependent | Strategy: Timestep-adaptive masking |
| Some heads consistently low entropy | Head specialization exists | Per-head selective masking |

### Next Steps (Phase 2):
1. Use the saved profiling data to design attention masks
2. Implement masking strategies from `2210/attention_profiling_and_masking_plan.md`
3. Run quality/efficiency experiments (FID, CLIP Score, LPIPS, speedup)

See `2210/attention_profiling.ipynb` for the standalone profiling notebook,
and `2210/attention_profiling_and_masking_plan.md` for the full experiment plan.
